## 📊 Modelo de sugerido de compras basadas en rotación de inventario.
#### 📁 sugerido de compras mensual por producto usando el historial de ventas y devoluciones.

In [0]:
from pyspark.sql.functions import (
    col,
    upper,
    trim,
    regexp_replace,
    expr,
    when,
    abs,
    sum,
    avg,
    round,
    first
)

# ==========================================
# LEER DATOS DESDE SILVER
# ==========================================
df = spark.table("workspace.silver.movcomercial")

# ==========================================
# NORMALIZAR CLASE
# ==========================================
df = df.withColumn(
    "CLASE",
    upper(trim(col("CLASE")))
)

# ==========================================
# LIMPIAR Y CONVERTIR CANTIDAD
# ==========================================
df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD"), r"\.", "")
)

df = df.withColumn(
    "CANTIDAD_TMP",
    regexp_replace(col("CANTIDAD_TMP"), ",", ".")
)

df = df.withColumn(
    "CANTIDAD_TMP",
    expr("try_cast(CANTIDAD_TMP as double)")
)

# ==========================================
# CREAR VENTA NETA
# FV00 = venta
# DV00 = devolución (ya viene negativa)
# ==========================================
df = df.withColumn(
    "VENTA_NETA_UNIDADES",
    when(
        col("CLASE") == "FV00",
        col("CANTIDAD_TMP")
    ).when(
        col("CLASE") == "DV00",
        col("CANTIDAD_TMP")
    ).otherwise(0)
)

# ==========================================
# AGRUPAR POR MES Y PRODUCTO
# ==========================================
ventas_mensuales = df.groupBy(
    "ANIO",
    "MES",
    "PRODUCTO"
).agg(

    first("PRODUCTONO")
    .alias("NOMBRE_PRODUCTO"),

    # Ventas netas en unidades
    round(
        sum("VENTA_NETA_UNIDADES"),
        2
    ).alias("VENTAS_NETAS_UNIDADES"),

    # Ventas brutas
    round(
        sum(
            when(
                col("CLASE") == "FV00",
                col("PARCIAL")
            ).otherwise(0)
        ),
        2
    ).alias("VENTAS_BRUTAS"),

    # Devoluciones
    round(
        abs(
            sum(
                when(
                    col("CLASE") == "DV00",
                    col("PARCIAL")
                ).otherwise(0)
            )
        ),
        2
    ).alias("DEVOLUCIONES")

)

# ==========================================
# PROMEDIO HISTÓRICO POR PRODUCTO
# ==========================================
promedio_producto = ventas_mensuales.groupBy(
    "PRODUCTO"
).agg(

    first("NOMBRE_PRODUCTO")
    .alias("NOMBRE_PRODUCTO"),

    round(
        avg("VENTAS_NETAS_UNIDADES"),
        2
    ).alias("PROMEDIO_MENSUAL")

)

# ==========================================
# CALCULAR STOCK DE SEGURIDAD
# ==========================================
modelo_compras = promedio_producto.withColumn(
    "STOCK_SEGURIDAD",
    round(
        col("PROMEDIO_MENSUAL") * 0.20,
        2
    )
)

# ==========================================
# SUGERIDO DE COMPRA
# ==========================================
modelo_compras = modelo_compras.withColumn(
    "SUGERIDO_COMPRA",
    round(
        col("PROMEDIO_MENSUAL") +
        col("STOCK_SEGURIDAD"),
        0
    )
)

# ==========================================
# ORDENAR RESULTADO
# ==========================================
modelo_compras = modelo_compras.orderBy(
    col("SUGERIDO_COMPRA").desc()
)

# ==========================================
# MOSTRAR RESULTADO
# ==========================================
display(modelo_compras)

# ==========================================
# CREAR ESQUEMA GOLD
# ==========================================
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
""")

# ==========================================
# GUARDAR MODELO EN GOLD
# ==========================================

modelo_compras.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.sugerido_compras")

print("Tabla workspace.gold.sugerido_compras creada correctamente")

# ==========================================
# VALIDAR TABLA FINAL
# ==========================================
df_gold = spark.table(
    "workspace.gold.sugerido_compras"
)

display(df_gold)

%md
## 📊 Modelo de sugerido de compras basadas en rotación de inventario.
#### 📁 Visualización gráfica del modelo - TOP 20 de referencias sugeridas en compra.

In [0]:
import matplotlib.pyplot as plt

# ==========================================
# LEER MODELO DESDE GOLD
# ==========================================
df = spark.table("workspace.gold.sugerido_compras")

# ==========================================
# TOP 20 PRODUCTOS
# ==========================================
top20 = df.orderBy(
    col("SUGERIDO_COMPRA").desc()
).limit(20)

# ==========================================
# CONVERTIR A PANDAS
# ==========================================
pdf = top20.toPandas()

# ==========================================
# GRAFICAR TORTA
# ==========================================
plt.figure(figsize=(12,12))

plt.pie(
    pdf["SUGERIDO_COMPRA"],
    labels=pdf["NOMBRE_PRODUCTO"],
    autopct='%1.1f%%'
)

plt.title("Top 20 Productos - Sugerido de Compras")
plt.tight_layout()
plt.show()